# Research Paper Ingestion & Processing

In [ ]:
%pip install arxiv langchain langchain_community pypdf

In [2]:
import arxiv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import tempfile
import time
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

## Text cleaning

In [3]:
def clean_text(text):
    """
    Cleans extracted research paper text:
    - Removes citations
    - Removes 'References' section
    - Cleans page numbers and headers
    - Removes LaTeX equations
    """
    text = re.sub(r"\[\d+\]", "", text)  # Remove citations like [15]
    text = re.sub(r"\(.*?\d{4}.*?\)", "", text)  # Remove inline citations
    text = re.sub(r"\bReferences\b.*", "", text, flags=re.DOTALL)  # Remove references section
    text = re.sub(r"Page\s\d+\sof\s\d+", "", text)  # Remove page numbers
    text = re.sub(r"\$.*?\$", "", text)  # Remove LaTeX math expressions
    text = re.sub(r"\s+", " ", text)  # Normalize spaces
    return text.strip()

## Fetching Research Papers from ArXiv
- Query ArXiv for academic papers based on a search term.
- Instead of storing PDFs, we load them **temporarily** then extract the text.

In [4]:
def fetch_papers(query="LLM", max_results=5):
    """
    Fetches academic papers from ArXiv based on a search term, processes the PDFs,
    extracts text, and splits the content into smaller, structured chunks.

    Parameters:
        query (str): The search term to query ArXiv for research papers.
        max_results (int): The maximum number of papers to retrieve.

    Returns:
        all_chunks: A list of document chunks extracted from the fetched papers.
    """
    
    # Initialize an ArXiv client
    client = arxiv.Client()

    # Search query with sorting (descending order to get the latest papers)
    search = arxiv.Search(query=query, max_results=max_results, sort_order=arxiv.SortOrder.Descending)
    
    all_chunks = []  # List to store extracted text chunks
    
    for result in client.results(search):
        # Display the title of the paper currently being processed
        print(f"Processing: {result.title}")
        
        # Create a temporary file to store the downloaded PDF
        temp_pdf = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
        try:
                result.download_pdf(filename=temp_pdf.name)
                print(f"-> Downloaded: {result.title}")
                temp_pdf.close()

                # Load PDF file 
                loader = PyPDFLoader(temp_pdf.name)
                papers = loader.load()
                
                # Combine all pages into a single string
                full_text = " ".join(p.page_content for p in papers)
                cleaned_text = clean_text(full_text)

                # Split the extracted text into chunks
                text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
                paper_chunks = text_splitter.create_documents([cleaned_text])
                
                # Add extracted chunks to the list
                all_chunks.extend(paper_chunks) 
                
        except (FileNotFoundError, ConnectionResetError) as e:
                print("Error occurred while downloading or processing PDF:", e)
                time.sleep(5)
    
    # Print total number of extracted chunks
    print(f"Total chunks extracted: {len(all_chunks)}")

    return all_chunks

## Exploratory Data Analysis

In [5]:
def analyze_text_chunks(chunks):
    num_chunks = len(chunks)
    chunk_lengths = [len(chunk.page_content.split()) for chunk in chunks]
    df = pd.DataFrame({"Chunk #": range(1, num_chunks+1), "Word Count": chunk_lengths})
    
    print("\nSummary Statistics for Chunk Sizes:")
    print(df.describe())
    
    plt.figure(figsize=(8,5))
    sns.histplot(df["Word Count"], bins=20, kde=True)
    plt.xlabel("Word Count per Chunk")
    plt.ylabel("Frequency")
    plt.title("Distribution of Word Counts in Chunks")
    plt.show()
    
    return df

# Function for common word analysis
def analyze_common_words(chunks, top_n=20):
    all_text = " ".join(chunk.page_content for chunk in chunks)
    words = re.findall(r'\b\w+\b', all_text.lower())
    word_counts = Counter(words)
    df_words = pd.DataFrame(word_counts.most_common(top_n), columns=["Word", "Frequency"])
    
    print("\nTop", top_n, "Most Common Words:")
    print(df_words)
    
    plt.figure(figsize=(10,5))
    sns.barplot(x=df_words["Frequency"], y=df_words["Word"])
    plt.xlabel("Frequency")
    plt.ylabel("Words")
    plt.title("Top " + str(top_n) + " Most Common Words")
    plt.show()
    
    return df_words


## Running the Paper Retrieval Pipeline
Testing the function by retrieving **3 recent papers** related to **Large Language Models**.

In [ ]:
query = "Large Language Models"
paper_chunks = fetch_papers(query=query, max_results=3)
df_chunks = analyze_text_chunks(paper_chunks)
df_words = analyze_common_words(paper_chunks)

for i, chunk in enumerate(paper_chunks[:3]):
    print(f"\nChunk {i+1}:\n{chunk.page_content[:300]}...")